# Audio Embeddings with Auditus


![](https://www.kaggle.com/competitions/91844/images/header)

[auditus](https://github.com/CarloLepelaars/auditus) is a new library for generating simple audio embeddings. In this notebook we will generate embeddings for the full training dataset using [Google's Bird Vocalization Classifier model](https://www.kaggle.com/models/google/bird-vocalization-classifier).

## Preparation

In [ ]:
!pip install -Uqq auditus

In [ ]:
import numpy as np
import pandas as pd
from os.path import relpath
from fastcore.all import *
from fastprogress.fastprogress import progress_bar
from fasttransform import Pipeline

from auditus.core import AudioArray
from auditus.transform import AudioLoader, Resampling, TFAudioEmbedding

Kaggle caps the Notebook output at 500 files so we will add `train_audio` embeddings as a column to the `train.csv` file.

In [ ]:
BASE_PATH = "/kaggle/input/birdclef-2025/"
train_soundscape_paths = np.random.choice(globtastic(f"{BASE_PATH}train_soundscapes", file_glob="*.ogg"), size=250)

In [ ]:
train_soundscape_paths[:3]

In [ ]:
df = pd.read_csv(f"{BASE_PATH}train.csv")
df.head(2)

In [ ]:
train_audio_paths = [f"{BASE_PATH}train_audio/{name}" for name in df['filename']]
print(len(train_audio_paths))
train_audio_paths[:3]

## Single instance

Let's look at a single example first.

The Bird vocalization model we use only works on $5$ seconds of data. For simplicity we truncate each file to the first $5$ seconds. The audio files have a sample rate of $32000$ so for $5$ seconds this equates to a length of $160000$.

In [ ]:
class Truncate(Transform):
    """ Get first 5 seconds for sample rate of 32000 and add padding if necessary. """
    def encodes(self, x:AudioArray): return AudioArray(np.pad(x.a, (0, max(0, 160000-len(x.a))))[:160000], x.sr)

In [ ]:
test_path = np.random.choice(train_audio_paths)
print(f"Test file: '{test_path}'")

Full file:

In [ ]:
full_audio = AudioLoader(sr=32000)(test_path)
full_audio.audio()

After truncation:

In [ ]:
truncated_audio = Truncate()(full_audio)
truncated_audio.audio()

## Pipeline

We start with a random audio file. The pipeline generates an embedding of length 1280. 

In [ ]:
pipe = Pipeline([AudioLoader(sr=32000), Truncate(), TFAudioEmbedding('/kaggle/input/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/')])

In [ ]:
output = pipe(test_path)
print(f"Embedding Length: '{output.shape}'")
print(f"First 5 elements of embedding: '{output[0][:5]}'")

## Full dataset

We are now ready to generate embeddings for the full training dataset. Using the GPU this takes around 10 minutes for the full training set.

In [ ]:
def get_emb(path): return pipe(path).squeeze(0).tolist()

with ThreadPoolExecutor() as ex:
    train_embs = list(progress_bar(ex.map(get_emb, train_audio_paths), total=len(train_audio_paths)))

In [ ]:
df["emb"] = train_embs

In [ ]:
df.to_csv("train_with_emb.csv")

The train file with embeddings is stored as a [Kaggle Dataset for easy exploration](https://www.kaggle.com/datasets/carlolepelaars/birdclef-2025-perch-embeddings). In [the next notebook](https://www.kaggle.com/code/carlolepelaars/birdclef-2025-embedding-visualization) we visualize these embeddings in 2 dimensions with [UMAP](https://umap-learn.readthedocs.io/en/latest) to get a good overview of the data and find patterns between the classes.

**That's it! Hope this helps you to get started with the competition!**

**If you like this Kaggle Notebook, consider giving an upvote and leaving a comment. Your feedback is very welcome! I will try to implement your suggestions.**